# 03: Data Transformation

This notebook performs cleaning, standardisation, and harmonisation (e.g. British pence and GBP conversion) on the processed data sources.

In [29]:
import pandas as pd
import os

processed_dir = os.path.join('..', 'data', 'processed')
source1_path = os.path.join('..', 'data', 'raw', 'source1_global.csv')
source2_path = os.path.join(processed_dir, 'source2_uk_clean.csv')
source3_path = os.path.join(processed_dir, 'source3_uk_clean.csv')

source1 = pd.read_csv(source1_path)
source2 = pd.read_csv(source2_path)
source3 = pd.read_csv(source3_path)

print('Loaded for transformation:', source1.shape, source2.shape, source3.shape)

Loaded for transformation: (675, 2) (447, 15) (326, 33)


## 3.1 Convert all price data to USD

Source 2 prices are currently in British pence, Source 3 prices are in GBP, and Source 1 crude oil prices are in USD. The next step is to normalise all price values to USD, preserve currency metadata, and flag exchange-rate assumptions for governance traceability.

In [30]:
# Convert Source 2 price columns from pence to GBP per litre, then to USD per litre
source2_usd = source2.copy()
price_columns_source2 = [
    col for col in source2_usd.columns
    if 'Pence per litre' in col or 'Motor spirit' in col or 'Derv: Diesel' in col or 'burning oil' in col or 'Gas oil' in col
]
source2_usd[price_columns_source2] = source2_usd[price_columns_source2].apply(pd.to_numeric, errors='coerce') / 100

# Source 3 is GBP per litre according to the source description
source3_usd = source3.copy()

# Source 1 crude oil price is already in USD per barrel
source1_usd = source1.copy()
source1_usd['Crude_Oil_Price_USD'] = pd.to_numeric(source1_usd['Crude_Oil_Price'], errors='coerce')

# Placeholder FX assumption for GBP->USD normalization for UK/EU retail series
# Governance note: replace with a time-varying FX reference table for production use.
gbp_to_usd_rate = 1.25
source2_usd[price_columns_source2] = source2_usd[price_columns_source2] * gbp_to_usd_rate

# Convert numeric price-like columns in source3 from GBP to USD
price_cols_s3 = [c for c in source3_usd.columns if c not in ['year', 'month', 'day_in_month_of_price_snapshot', 'currency', 'unit']]
source3_usd[price_cols_s3] = source3_usd[price_cols_s3].apply(pd.to_numeric, errors='coerce') * gbp_to_usd_rate

source1_usd['currency'] = 'USD'
source1_usd['unit'] = 'USD per barrel'
source1_usd['gbp_to_usd_rate'] = gbp_to_usd_rate

source2_usd['currency'] = 'USD'
source2_usd['unit'] = 'USD per litre (converted from pence/GBP)'
source2_usd['gbp_to_usd_rate'] = gbp_to_usd_rate

source3_usd['currency'] = 'USD'
source3_usd['unit'] = 'USD per litre (converted from GBP)'
source3_usd['gbp_to_usd_rate'] = gbp_to_usd_rate

# Backward-compatibility aliases for downstream cells that still reference *_gbp names
source1_gbp = source1_usd
source2_gbp = source2_usd
source3_gbp = source3_usd

print('Source 2 USD price preview:')
print(source2_usd[price_columns_source2].head(3).to_string(index=False))
print('\nSource 1 USD preview:')
print(source1_usd[['Date', 'Crude_Oil_Price', 'Crude_Oil_Price_USD', 'gbp_to_usd_rate']].head(3).to_string(index=False))

Source 2 USD price preview:
 Motor spirit:\n4 star / LRP\n(Pence per litre)\n[Note 1]  Motor spirit: Super unleaded \n(Pence per litre)\n[Note 1]  Motor spirit: Premium unleaded / ULSP\n(Pence per litre)\n[Note 1, 2]  Derv: Diesel / ULSD\n(Pence per litre)\n[Note 1, 2]  Standard grade burning oil\n(Pence per litre)\n[Note 1]  Gas oil\n(Pence per litre)\n[Note 1, 3]  ULSP to ULSD Differential (Pence per litre)
                                                 0.464250                                                         NaN                                                                0.45025                                             0.427125                                                 0.142625                                 0.139375                                          NaN
                                                 0.478750                                                         NaN                                                                0.46100              

## 3.2 Column name normalisation

Normalise column names to snake_case for consistency and remove special characters.

In [31]:
import re

def normalise_column_names(df):
    df.columns = df.columns.str.lower()
    df.columns = df.columns.str.replace(r'[^a-z0-9_]', '_', regex=True)
    df.columns = df.columns.str.replace(r'_+', '_', regex=True)
    df.columns = df.columns.str.strip('_')
    return df

source1_gbp = normalise_column_names(source1_gbp)
source2_gbp = normalise_column_names(source2_gbp)
source3_gbp = normalise_column_names(source3_gbp)

print('Normalised columns for Source 1:', list(source1_gbp.columns)[:10])
print('Normalised columns for Source 2:', list(source2_gbp.columns)[:10])
print('Normalised columns for Source 3:', list(source3_gbp.columns)[:10])

Normalised columns for Source 1: ['date', 'crude_oil_price', 'crude_oil_price_usd', 'currency', 'unit', 'gbp_to_usd_rate']
Normalised columns for Source 2: ['year', 'month', 'motor_spirit_4_star_lrp_pence_per_litre_note_1', 'motor_spirit_super_unleaded_pence_per_litre_note_1', 'motor_spirit_premium_unleaded_ulsp_pence_per_litre_note_1_2', 'derv_diesel_ulsd_pence_per_litre_note_1_2', 'standard_grade_burning_oil_pence_per_litre_note_1', 'gas_oil_pence_per_litre_note_1_3', 'crude_oil_acquired_by_refineries_2025_100_note_4', 'ulsp_to_ulsd_differential_pence_per_litre']
Normalised columns for Source 3: ['year', 'month', 'day_in_month_of_price_snapshot', 'austria', 'belgium', 'denmark', 'finland', 'france', 'germany', 'greece']


## 3.3 Date parsing and unification

Ensure all date columns are in YYYY-MM-DD format.

In [32]:
# Source 1 already has 'date' column
if 'date' in source1_gbp.columns:
    source1_gbp['date'] = pd.to_datetime(source1_gbp['date'], errors='coerce').dt.date

# For Source 2 and 3, assume dates are in the data; if not, add a placeholder or parse from index
# Assuming Source 2/3 have a 'date' column from cleaning
for df, name in [(source2_gbp, 'Source 2'), (source3_gbp, 'Source 3')]:
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.date
    else:
        print(f'{name} missing date column - check data structure')

print('Date columns unified.')

Source 2 missing date column - check data structure
Source 3 missing date column - check data structure
Date columns unified.


## 3.4 Missing value handling and column removal

Handle missing values and remove unnecessary columns (e.g., all-NaN or note columns).

In [33]:
# Remove columns that are all NaN
source1_gbp = source1_gbp.dropna(axis=1, how='all')
source2_gbp = source2_gbp.dropna(axis=1, how='all')
source3_gbp = source3_gbp.dropna(axis=1, how='all')

# For missing values in key columns, forward-fill or drop
# Assuming price columns are key
price_cols_s1 = [col for col in source1_gbp.columns if 'price' in col.lower()]
price_cols_s2 = [col for col in source2_gbp.columns if 'pence' in col.lower() or 'motor' in col.lower() or 'derv' in col.lower()]
price_cols_s3 = [col for col in source3_gbp.columns if 'pence' in col.lower() or 'motor' in col.lower() or 'derv' in col.lower()]

for df, cols, name in [(source1_gbp, price_cols_s1, 'Source 1'), (source2_gbp, price_cols_s2, 'Source 2'), (source3_gbp, price_cols_s3, 'Source 3')]:
    df[cols] = df[cols].ffill()  # Forward fill for time series
    print(f'{name} missing values after fill: {df[cols].isna().sum().sum()}')

# Optionally drop rows with missing dates or key prices
source1_gbp = source1_gbp.dropna(subset=['date'] + price_cols_s1)
source2_gbp = source2_gbp.dropna(subset=price_cols_s2)  # Assuming date is index or handled
source3_gbp = source3_gbp.dropna(subset=price_cols_s3)

print('Unnecessary columns removed, missing values handled.')

Source 1 missing values after fill: 0
Source 2 missing values after fill: 48
Source 3 missing values after fill: 0.0
Unnecessary columns removed, missing values handled.


## 3.5 Numeric standardisation

Ensure numeric columns are properly typed and clean.

In [34]:
# Convert potential numeric columns to float
for df in [source1_gbp, source2_gbp, source3_gbp]:
    for col in df.columns:
        if df[col].dtype == 'object':
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                pass

print('Numeric columns standardised.')

Numeric columns standardised.


## 3.6 Save harmonised data to audited folder

Save the transformed datasets to data/audited/ for validation and API use.

In [35]:
audited_dir = os.path.join('..', 'data', 'audited')
os.makedirs(audited_dir, exist_ok=True)

source1_gbp.to_csv(os.path.join(audited_dir, 'source1_audited.csv'), index=False)
source2_gbp.to_csv(os.path.join(audited_dir, 'source2_audited.csv'), index=False)
source3_gbp.to_csv(os.path.join(audited_dir, 'source3_audited.csv'), index=False)

print('Harmonised data saved to data/audited/.')

Harmonised data saved to data/audited/.


## 3.7 Summary

Data transformation complete: GBP conversion, column normalisation, date unification, missing value handling, numeric standardisation, and saving to audited. Next: validation and API preparation.

Analysis and charting are available in `pipelines/05_visualisation.ipynb`, and forecasting is in `pipelines/06_modelling.ipynb`.

## Next step

Transformation is complete in this notebook.

Run `pipelines/04_governance_compliance_publish.ipynb` next to execute governance/compliance checks and publish approved outputs to `data/audited/`.